# Tutorial: G25 Part A - Standalone AdView Workspace, Confounds, EV Files, FEAT, and MRIcroGL

This notebook is written to run in a clean Neurodesk working directory without `git clone` or `git pull`.
It creates a local `g25/` folder, downloads the required `ds007486` `v1.0.0` AdView files directly from OpenNeuro,
builds FEAT-ready motion confounds for one representative run, converts all available AdView `events.tsv` files
into FSL 3-column EV files, runs one representative first-level FEAT model, and renders the result with MRIcroGL.


## Outline

1. Create or reuse a local `g25/` workspace in the current directory.
2. Download the AdView files needed for this homework directly from OpenNeuro.
3. Inspect a representative anatomy file, AdView BOLD run, and AdView event structure.
4. Build FEAT-ready motion confounds from `mcflirt` for one representative AdView run.
5. Convert all available AdView `events.tsv` files to FSL 3-column EV files.
6. Run one representative first-level FEAT model for AdView.
7. Visualize the FEAT result with MRIcroGL and embed the screenshot inline.


In [ ]:
from __future__ import annotations

import gzip
import importlib.util
import json
import os
import shutil
import ssl
import struct
import subprocess
import sys
import textwrap
import urllib.request
from pathlib import Path

required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "nibabel": "nibabel",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise RuntimeError(
        "This notebook expects the current Jupyter kernel to already provide: "
        + ", ".join(missing)
    )

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

plt.rcParams["figure.dpi"] = 120
CTX = ssl._create_unverified_context()
OPENNEURO_API = "https://openneuro.org/crn/graphql"
DATASET_ID = "ds007486"
SNAPSHOT_TAG = "1.0.0"
APPROVED_SUBJECTS = ["sub-10562", "sub-10665", "sub-11028", "sub-11039", "sub-11450"]
REPRESENTATIVE_SUBJECT = "sub-11039"
TASK = "adview"
RUN = 1
ECHO = 1
PROJECT_ROOT_OVERRIDE = os.environ.get("G25_PROJECT_ROOT", "").strip()
WORKSPACE_README = """# G25

Standalone AdView workspace created by this tutorial notebook.
This folder is intentionally lightweight and is designed to be created directly in Neurodesk without cloning a repository.
"""
CODE_README = """# G25 Standalone Notebook Workspace

This folder was created by the AdView tutorial notebook.
The notebook writes BIDS task metadata, downloads AdView events from OpenNeuro, generates FSL EV files, and runs a representative first-level FEAT model.
"""
TEMPLATE_ADVIEW_FSF = '\n# FEAT version number\nset fmri(version) 6.00\n\n# Are we in MELODIC?\nset fmri(inmelodic) 0\n\n# Analysis level\n# 1 : First-level analysis\n# 2 : Higher-level analysis\nset fmri(level) 1\n\n# Which stages to run\n# 0 : No first-level analysis (registration and/or group stats only)\n# 7 : Full first-level analysis\n# 1 : Pre-processing\n# 2 : Statistics\nset fmri(analysis) 7\n\n# Use relative filenames\nset fmri(relative_yn) 0\n\n# Balloon help\nset fmri(help_yn) 1\n\n# Run Featwatcher\nset fmri(featwatcher_yn) 0\n\n# Cleanup first-level standard-space images\nset fmri(sscleanup_yn) 0\n\n# Output directory\nset fmri(outputdir) "OUTPUT"\n\n# TR(s)\nset fmri(tr) 1.615000\n\n# Total volumes\nset fmri(npts) NVOLS\n\n# Delete volumes\nset fmri(ndelete) 0\n\n# Perfusion tag/control order\nset fmri(tagfirst) 1\n\n# Number of first-level analyses\nset fmri(multiple) 1\n\n# Higher-level input type\n# 1 : Inputs are lower-level FEAT directories\n# 2 : Inputs are cope images from FEAT directories\nset fmri(inputtype) 2\n\n# Carry out pre-stats processing?\nset fmri(filtering_yn) 1\n\n# Brain/background threshold, %\nset fmri(brain_thresh) 10\n\n# Critical z for design efficiency calculation\nset fmri(critical_z) 5.3\n\n# Noise level\nset fmri(noise) 0.66\n\n# Noise AR(1)\nset fmri(noisear) 0.34\n\n# Motion correction\n# 0 : None\n# 1 : MCFLIRT\nset fmri(mc) 0\n\n# Spin-history (currently obsolete)\nset fmri(sh_yn) 0\n\n# B0 fieldmap unwarping?\nset fmri(regunwarp_yn) 0\n\n# GDC Test\nset fmri(gdc) ""\n\n# EPI dwell time (ms)\nset fmri(dwell) 0.7\n\n# EPI TE (ms)\nset fmri(te) 35\n\n# % Signal loss threshold\nset fmri(signallossthresh) 10\n\n# Unwarp direction\nset fmri(unwarp_dir) y-\n\n# Slice timing correction\n# 0 : None\n# 1 : Regular up (0, 1, 2, 3, ...)\n# 2 : Regular down\n# 3 : Use slice order file\n# 4 : Use slice timings file\n# 5 : Interleaved (0, 2, 4 ... 1, 3, 5 ... )\nset fmri(st) 0\n\n# Slice timings file\nset fmri(st_file) ""\n\n# BET brain extraction\nset fmri(bet_yn) 1\n\n# Spatial smoothing FWHM (mm)\nset fmri(smooth) "SMOOTH"\n\n# Intensity normalization\nset fmri(norm_yn) 0\n\n# Perfusion subtraction\nset fmri(perfsub_yn) 0\n\n# Highpass temporal filtering\nset fmri(temphp_yn) 0\n\n# Lowpass temporal filtering\nset fmri(templp_yn) 0\n\n# MELODIC ICA data exploration\nset fmri(melodic_yn) 0\n\n# Carry out main stats?\nset fmri(stats_yn) 1\n\n# Carry out prewhitening?\nset fmri(prewhiten_yn) 1\n\n# Add motion parameters to model\n# 0 : No\n# 1 : Yes\nset fmri(motionevs) 0\nset fmri(motionevsbeta) ""\nset fmri(scriptevsbeta) ""\n\n# Robust outlier detection in FLAME?\nset fmri(robust_yn) 0\n\n# Higher-level modelling\n# 3 : Fixed effects\n# 0 : Mixed Effects: Simple OLS\n# 2 : Mixed Effects: FLAME 1\n# 1 : Mixed Effects: FLAME 1+2\nset fmri(mixed_yn) 2\n\n# Higher-level permutations\nset fmri(randomisePermutations) 5000\n\n# Number of EVs\nset fmri(evs_orig) 5\nset fmri(evs_real) 5\nset fmri(evs_vox) 0\n\n# Number of contrasts\nset fmri(ncon_orig) 7\nset fmri(ncon_real) 7\n\n# Number of F-tests\nset fmri(nftests_orig) 0\nset fmri(nftests_real) 0\n\n# Add constant column to design matrix? (obsolete)\nset fmri(constcol) 0\n\n# Carry out post-stats steps?\nset fmri(poststats_yn) 1\n\n# Pre-threshold masking?\nset fmri(threshmask) ""\n\n# Thresholding\n# 0 : None\n# 1 : Uncorrected\n# 2 : Voxel\n# 3 : Cluster\nset fmri(thresh) 3\n\n# P threshold\nset fmri(prob_thresh) 0.05\n\n# Z threshold\nset fmri(z_thresh) 3.1000000000000005\n\n# Z min/max for colour rendering\n# 0 : Use actual Z min/max\n# 1 : Use preset Z min/max\nset fmri(zdisplay) 0\n\n# Z min in colour rendering\nset fmri(zmin) 2\n\n# Z max in colour rendering\nset fmri(zmax) 8\n\n# Colour rendering type\n# 0 : Solid blobs\n# 1 : Transparent blobs\nset fmri(rendertype) 1\n\n# Background image for higher-level stats overlays\n# 1 : Mean highres\n# 2 : First highres\n# 3 : Mean functional\n# 4 : First functional\n# 5 : Standard space template\nset fmri(bgimage) 1\n\n# Create time series plots\nset fmri(tsplot_yn) 1\n\n# Registration to initial structural\nset fmri(reginitial_highres_yn) 0\n\n# Search space for registration to initial structural\n# 0   : No search\n# 90  : Normal search\n# 180 : Full search\nset fmri(reginitial_highres_search) 90\n\n# Degrees of Freedom for registration to initial structural\nset fmri(reginitial_highres_dof) 3\n\n# Registration to main structural\nset fmri(reghighres_yn) 0\n\n# Search space for registration to main structural\n# 0   : No search\n# 90  : Normal search\n# 180 : Full search\nset fmri(reghighres_search) 90\n\n# Degrees of Freedom for registration to main structural\nset fmri(reghighres_dof) BBR\n\n# Registration to standard image?\nset fmri(regstandard_yn) 0\n\n# Use alternate reference images?\nset fmri(alternateReference_yn) 0\n\n# Standard image\nset fmri(regstandard) "/usr/share/fsl/5.0/data/standard/MNI152_T1_2mm_brain"\n\n# Search space for registration to standard space\n# 0   : No search\n# 90  : Normal search\n# 180 : Full search\nset fmri(regstandard_search) 90\n\n# Degrees of Freedom for registration to standard space\nset fmri(regstandard_dof) 12\n\n# Do nonlinear registration from structural to standard space?\nset fmri(regstandard_nonlinear_yn) 0\n\n# Control nonlinear warp field resolution\nset fmri(regstandard_nonlinear_warpres) 10\n\n# High pass filter cutoff\nset fmri(paradigm_hp) 200\n\n# Total voxels\nset fmri(totalVoxels) 71101800\n\n\n# Number of lower-level copes feeding into higher-level analysis\nset fmri(ncopeinputs) 0\n\n# 4D AVW data or FEAT directory (1)\nset feat_files(1) "DATA"\n\n# Add confound EVs text file\nset fmri(confoundevs) 1\n\n# Confound EVs text file for analysis 1\nset confoundev_files(1) "CONFOUNDEVS"\n\n# EV 1 title\nset fmri(evtitle1) "basic"\n\n# Basic waveform shape (EV 1)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape1) 3\n\n# Convolution (EV 1)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve1) 3\n\n# Convolve phase (EV 1)\nset fmri(convolve_phase1) 0\n\n# Apply temporal filtering (EV 1)\nset fmri(tempfilt_yn1) 0\n\n# Add temporal derivative (EV 1)\nset fmri(deriv_yn1) 0\n\n# Custom EV file (EV 1)\nset fmri(custom1) "EVDIR_Every_day_products.txt"\n\n# Orthogonalise EV 1 wrt EV 0\nset fmri(ortho1.0) 0\n\n# Orthogonalise EV 1 wrt EV 1\nset fmri(ortho1.1) 0\n\n# Orthogonalise EV 1 wrt EV 2\nset fmri(ortho1.2) 0\n\n# Orthogonalise EV 1 wrt EV 3\nset fmri(ortho1.3) 0\n\n# Orthogonalise EV 1 wrt EV 4\nset fmri(ortho1.4) 0\n\n# Orthogonalise EV 1 wrt EV 5\nset fmri(ortho1.5) 0\n\n# EV 2 title\nset fmri(evtitle2) "gambling"\n\n# Basic waveform shape (EV 2)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape2) 3\n\n# Convolution (EV 2)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve2) 3\n\n# Convolve phase (EV 2)\nset fmri(convolve_phase2) 0\n\n# Apply temporal filtering (EV 2)\nset fmri(tempfilt_yn2) 0\n\n# Add temporal derivative (EV 2)\nset fmri(deriv_yn2) 0\n\n# Custom EV file (EV 2)\nset fmri(custom2) "EVDIR_Gambling.txt"\n\n# Orthogonalise EV 2 wrt EV 0\nset fmri(ortho2.0) 0\n\n# Orthogonalise EV 2 wrt EV 1\nset fmri(ortho2.1) 0\n\n# Orthogonalise EV 2 wrt EV 2\nset fmri(ortho2.2) 0\n\n# Orthogonalise EV 2 wrt EV 3\nset fmri(ortho2.3) 0\n\n# Orthogonalise EV 2 wrt EV 4\nset fmri(ortho2.4) 0\n\n# Orthogonalise EV 2 wrt EV 5\nset fmri(ortho2.5) 0\n\n# EV 3 title\nset fmri(evtitle3) "vmpfc"\n\n# Basic waveform shape (EV 3)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape3) 3\n\n# Convolution (EV 3)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve3) 3\n\n# Convolve phase (EV 3)\nset fmri(convolve_phase3) 0\n\n# Apply temporal filtering (EV 3)\nset fmri(tempfilt_yn3) 0\n\n# Add temporal derivative (EV 3)\nset fmri(deriv_yn3) 0\n\n# Custom EV file (EV 3)\nset fmri(custom3) "EVDIR_vmPFC.txt"\n\n# Orthogonalise EV 3 wrt EV 0\nset fmri(ortho3.0) 0\n\n# Orthogonalise EV 3 wrt EV 1\nset fmri(ortho3.1) 0\n\n# Orthogonalise EV 3 wrt EV 2\nset fmri(ortho3.2) 0\n\n# Orthogonalise EV 3 wrt EV 3\nset fmri(ortho3.3) 0\n\n# Orthogonalise EV 3 wrt EV 4\nset fmri(ortho3.4) 0\n\n# Orthogonalise EV 3 wrt EV 5\nset fmri(ortho3.5) 0\n\n# EV 4 title\nset fmri(evtitle4) "attn"\n\n# Basic waveform shape (EV 4)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape4) 3\n\n# Convolution (EV 4)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve4) 3\n\n# Convolve phase (EV 4)\nset fmri(convolve_phase4) 0\n\n# Apply temporal filtering (EV 4)\nset fmri(tempfilt_yn4) 0\n\n# Add temporal derivative (EV 4)\nset fmri(deriv_yn4) 0\n\n# Custom EV file (EV 4)\nset fmri(custom4) "EVDIR_attention_check.txt"\n\n# Orthogonalise EV 4 wrt EV 0\nset fmri(ortho4.0) 0\n\n# Orthogonalise EV 4 wrt EV 1\nset fmri(ortho4.1) 0\n\n# Orthogonalise EV 4 wrt EV 2\nset fmri(ortho4.2) 0\n\n# Orthogonalise EV 4 wrt EV 3\nset fmri(ortho4.3) 0\n\n# Orthogonalise EV 4 wrt EV 4\nset fmri(ortho4.4) 0\n\n# Orthogonalise EV 4 wrt EV 5\nset fmri(ortho4.5) 0\n\n# EV 5 title\nset fmri(evtitle5) "MISSED_TRIAL"\n\n# Basic waveform shape (EV 5)\n# 0 : Square\n# 1 : Sinusoid\n# 2 : Custom (1 entry per volume)\n# 3 : Custom (3 column format)\n# 4 : Interaction\n# 10 : Empty (all zeros)\nset fmri(shape5) EV_SHAPE\n\n# Convolution (EV 5)\n# 0 : None\n# 1 : Gaussian\n# 2 : Gamma\n# 3 : Double-Gamma HRF\n# 4 : Gamma basis functions\n# 5 : Sine basis functions\n# 6 : FIR basis functions\n# 8 : Alternate Double-Gamma\nset fmri(convolve5) 3\n\n# Convolve phase (EV 5)\nset fmri(convolve_phase5) 0\n\n# Apply temporal filtering (EV 5)\nset fmri(tempfilt_yn5) 0\n\n# Add temporal derivative (EV 5)\nset fmri(deriv_yn5) 0\n\n# Custom EV file (EV 5)\nset fmri(custom5) "EVDIR_missed_trial.txt"\n\n# Orthogonalise EV 5 wrt EV 0\nset fmri(ortho5.0) 0\n\n# Orthogonalise EV 5 wrt EV 1\nset fmri(ortho5.1) 0\n\n# Orthogonalise EV 5 wrt EV 2\nset fmri(ortho5.2) 0\n\n# Orthogonalise EV 5 wrt EV 3\nset fmri(ortho5.3) 0\n\n# Orthogonalise EV 5 wrt EV 4\nset fmri(ortho5.4) 0\n\n# Orthogonalise EV 5 wrt EV 5\nset fmri(ortho5.5) 0\n\n# Contrast & F-tests mode\n# real : control real EVs\n# orig : control original EVs\nset fmri(con_mode_old) orig\nset fmri(con_mode) orig\n\n# Display images for contrast_real 1\nset fmri(conpic_real.1) 1\n\n# Title for contrast_real 1\nset fmri(conname_real.1) "basic"\n\n# Real contrast_real vector 1 element 1\nset fmri(con_real1.1) 1\n\n# Real contrast_real vector 1 element 2\nset fmri(con_real1.2) 0\n\n# Real contrast_real vector 1 element 3\nset fmri(con_real1.3) 0\n\n# Real contrast_real vector 1 element 4\nset fmri(con_real1.4) 0\n\n# Real contrast_real vector 1 element 5\nset fmri(con_real1.5) 0\n\n# Display images for contrast_real 2\nset fmri(conpic_real.2) 1\n\n# Title for contrast_real 2\nset fmri(conname_real.2) "gambling"\n\n# Real contrast_real vector 2 element 1\nset fmri(con_real2.1) 0\n\n# Real contrast_real vector 2 element 2\nset fmri(con_real2.2) 1\n\n# Real contrast_real vector 2 element 3\nset fmri(con_real2.3) 0\n\n# Real contrast_real vector 2 element 4\nset fmri(con_real2.4) 0\n\n# Real contrast_real vector 2 element 5\nset fmri(con_real2.5) 0\n\n# Display images for contrast_real 3\nset fmri(conpic_real.3) 1\n\n# Title for contrast_real 3\nset fmri(conname_real.3) "vmPFC"\n\n# Real contrast_real vector 3 element 1\nset fmri(con_real3.1) 0\n\n# Real contrast_real vector 3 element 2\nset fmri(con_real3.2) 0\n\n# Real contrast_real vector 3 element 3\nset fmri(con_real3.3) 1\n\n# Real contrast_real vector 3 element 4\nset fmri(con_real3.4) 0\n\n# Real contrast_real vector 3 element 5\nset fmri(con_real3.5) 0\n\n# Display images for contrast_real 4\nset fmri(conpic_real.4) 1\n\n# Title for contrast_real 4\nset fmri(conname_real.4) "g > b"\n\n# Real contrast_real vector 4 element 1\nset fmri(con_real4.1) -1.0\n\n# Real contrast_real vector 4 element 2\nset fmri(con_real4.2) 1.0\n\n# Real contrast_real vector 4 element 3\nset fmri(con_real4.3) 0\n\n# Real contrast_real vector 4 element 4\nset fmri(con_real4.4) 0.0\n\n# Real contrast_real vector 4 element 5\nset fmri(con_real4.5) 0\n\n# Display images for contrast_real 5\nset fmri(conpic_real.5) 1\n\n# Title for contrast_real 5\nset fmri(conname_real.5) "b > vmPFC"\n\n# Real contrast_real vector 5 element 1\nset fmri(con_real5.1) 1.0\n\n# Real contrast_real vector 5 element 2\nset fmri(con_real5.2) 0.0\n\n# Real contrast_real vector 5 element 3\nset fmri(con_real5.3) -1.0\n\n# Real contrast_real vector 5 element 4\nset fmri(con_real5.4) 0\n\n# Real contrast_real vector 5 element 5\nset fmri(con_real5.5) 0.0\n\n# Display images for contrast_real 6\nset fmri(conpic_real.6) 1\n\n# Title for contrast_real 6\nset fmri(conname_real.6) "g > vmPFC"\n\n# Real contrast_real vector 6 element 1\nset fmri(con_real6.1) 0.0\n\n# Real contrast_real vector 6 element 2\nset fmri(con_real6.2) 1.0\n\n# Real contrast_real vector 6 element 3\nset fmri(con_real6.3) -1.0\n\n# Real contrast_real vector 6 element 4\nset fmri(con_real6.4) 0\n\n# Real contrast_real vector 6 element 5\nset fmri(con_real6.5) 0\n\n# Display images for contrast_real 7\nset fmri(conpic_real.7) 1\n\n# Title for contrast_real 7\nset fmri(conname_real.7) "g > (b + vmPFC)"\n\n# Real contrast_real vector 7 element 1\nset fmri(con_real7.1) -1.0\n\n# Real contrast_real vector 7 element 2\nset fmri(con_real7.2) 2.0\n\n# Real contrast_real vector 7 element 3\nset fmri(con_real7.3) -1.0\n\n# Real contrast_real vector 7 element 4\nset fmri(con_real7.4) 0\n\n# Real contrast_real vector 7 element 5\nset fmri(con_real7.5) 0\n\n# Display images for contrast_orig 1\nset fmri(conpic_orig.1) 1\n\n# Title for contrast_orig 1\nset fmri(conname_orig.1) "basic"\n\n# Real contrast_orig vector 1 element 1\nset fmri(con_orig1.1) 1\n\n# Real contrast_orig vector 1 element 2\nset fmri(con_orig1.2) 0\n\n# Real contrast_orig vector 1 element 3\nset fmri(con_orig1.3) 0\n\n# Real contrast_orig vector 1 element 4\nset fmri(con_orig1.4) 0\n\n# Real contrast_orig vector 1 element 5\nset fmri(con_orig1.5) 0\n\n# Display images for contrast_orig 2\nset fmri(conpic_orig.2) 1\n\n# Title for contrast_orig 2\nset fmri(conname_orig.2) "gambling"\n\n# Real contrast_orig vector 2 element 1\nset fmri(con_orig2.1) 0\n\n# Real contrast_orig vector 2 element 2\nset fmri(con_orig2.2) 1\n\n# Real contrast_orig vector 2 element 3\nset fmri(con_orig2.3) 0\n\n# Real contrast_orig vector 2 element 4\nset fmri(con_orig2.4) 0\n\n# Real contrast_orig vector 2 element 5\nset fmri(con_orig2.5) 0\n\n# Display images for contrast_orig 3\nset fmri(conpic_orig.3) 1\n\n# Title for contrast_orig 3\nset fmri(conname_orig.3) "vmPFC"\n\n# Real contrast_orig vector 3 element 1\nset fmri(con_orig3.1) 0\n\n# Real contrast_orig vector 3 element 2\nset fmri(con_orig3.2) 0\n\n# Real contrast_orig vector 3 element 3\nset fmri(con_orig3.3) 1\n\n# Real contrast_orig vector 3 element 4\nset fmri(con_orig3.4) 0\n\n# Real contrast_orig vector 3 element 5\nset fmri(con_orig3.5) 0\n\n# Display images for contrast_orig 4\nset fmri(conpic_orig.4) 1\n\n# Title for contrast_orig 4\nset fmri(conname_orig.4) "g > b"\n\n# Real contrast_orig vector 4 element 1\nset fmri(con_orig4.1) -1.0\n\n# Real contrast_orig vector 4 element 2\nset fmri(con_orig4.2) 1.0\n\n# Real contrast_orig vector 4 element 3\nset fmri(con_orig4.3) 0\n\n# Real contrast_orig vector 4 element 4\nset fmri(con_orig4.4) 0.0\n\n# Real contrast_orig vector 4 element 5\nset fmri(con_orig4.5) 0\n\n# Display images for contrast_orig 5\nset fmri(conpic_orig.5) 1\n\n# Title for contrast_orig 5\nset fmri(conname_orig.5) "b > vmPFC"\n\n# Real contrast_orig vector 5 element 1\nset fmri(con_orig5.1) 1.0\n\n# Real contrast_orig vector 5 element 2\nset fmri(con_orig5.2) 0.0\n\n# Real contrast_orig vector 5 element 3\nset fmri(con_orig5.3) -1.0\n\n# Real contrast_orig vector 5 element 4\nset fmri(con_orig5.4) 0\n\n# Real contrast_orig vector 5 element 5\nset fmri(con_orig5.5) 0.0\n\n# Display images for contrast_orig 6\nset fmri(conpic_orig.6) 1\n\n# Title for contrast_orig 6\nset fmri(conname_orig.6) "g > vmPFC"\n\n# Real contrast_orig vector 6 element 1\nset fmri(con_orig6.1) 0.0\n\n# Real contrast_orig vector 6 element 2\nset fmri(con_orig6.2) 1.0\n\n# Real contrast_orig vector 6 element 3\nset fmri(con_orig6.3) -1.0\n\n# Real contrast_orig vector 6 element 4\nset fmri(con_orig6.4) 0\n\n# Real contrast_orig vector 6 element 5\nset fmri(con_orig6.5) 0\n\n# Display images for contrast_orig 7\nset fmri(conpic_orig.7) 1\n\n# Title for contrast_orig 7\nset fmri(conname_orig.7) "g > (b + vmPFC)"\n\n# Real contrast_orig vector 7 element 1\nset fmri(con_orig7.1) -1.0\n\n# Real contrast_orig vector 7 element 2\nset fmri(con_orig7.2) 2.0\n\n# Real contrast_orig vector 7 element 3\nset fmri(con_orig7.3) -1.0\n\n# Real contrast_orig vector 7 element 4\nset fmri(con_orig7.4) 0\n\n# Real contrast_orig vector 7 element 5\nset fmri(con_orig7.5) 0\n\n# Contrast masking - use >0 instead of thresholding?\nset fmri(conmask_zerothresh_yn) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 2?\nset fmri(conmask1_2) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 3?\nset fmri(conmask1_3) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 4?\nset fmri(conmask1_4) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 5?\nset fmri(conmask1_5) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 6?\nset fmri(conmask1_6) 0\n\n# Mask real contrast/F-test 1 with real contrast/F-test 7?\nset fmri(conmask1_7) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 1?\nset fmri(conmask2_1) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 3?\nset fmri(conmask2_3) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 4?\nset fmri(conmask2_4) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 5?\nset fmri(conmask2_5) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 6?\nset fmri(conmask2_6) 0\n\n# Mask real contrast/F-test 2 with real contrast/F-test 7?\nset fmri(conmask2_7) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 1?\nset fmri(conmask3_1) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 2?\nset fmri(conmask3_2) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 4?\nset fmri(conmask3_4) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 5?\nset fmri(conmask3_5) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 6?\nset fmri(conmask3_6) 0\n\n# Mask real contrast/F-test 3 with real contrast/F-test 7?\nset fmri(conmask3_7) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 1?\nset fmri(conmask4_1) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 2?\nset fmri(conmask4_2) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 3?\nset fmri(conmask4_3) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 5?\nset fmri(conmask4_5) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 6?\nset fmri(conmask4_6) 0\n\n# Mask real contrast/F-test 4 with real contrast/F-test 7?\nset fmri(conmask4_7) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 1?\nset fmri(conmask5_1) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 2?\nset fmri(conmask5_2) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 3?\nset fmri(conmask5_3) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 4?\nset fmri(conmask5_4) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 6?\nset fmri(conmask5_6) 0\n\n# Mask real contrast/F-test 5 with real contrast/F-test 7?\nset fmri(conmask5_7) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 1?\nset fmri(conmask6_1) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 2?\nset fmri(conmask6_2) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 3?\nset fmri(conmask6_3) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 4?\nset fmri(conmask6_4) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 5?\nset fmri(conmask6_5) 0\n\n# Mask real contrast/F-test 6 with real contrast/F-test 7?\nset fmri(conmask6_7) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 1?\nset fmri(conmask7_1) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 2?\nset fmri(conmask7_2) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 3?\nset fmri(conmask7_3) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 4?\nset fmri(conmask7_4) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 5?\nset fmri(conmask7_5) 0\n\n# Mask real contrast/F-test 7 with real contrast/F-test 6?\nset fmri(conmask7_6) 0\n\n# Do contrast masking at all?\nset fmri(conmask1_1) 0\n\n##########################################################\n# Now options that don\'t appear in the GUI\n\n# Alternative (to BETting) mask image\nset fmri(alternative_mask) ""\n\n# Initial structural space registration initialisation transform\nset fmri(init_initial_highres) ""\n\n# Structural space registration initialisation transform\nset fmri(init_highres) ""\n\n# Standard space registration initialisation transform\nset fmri(init_standard) ""\n\n# For full FEAT analysis: overwrite existing .feat output dir?\nset fmri(overwrite_yn) 1\n'


def find_or_create_project_root(start: Path | None = None, override: str | Path | None = None) -> Path:
    candidates: list[Path] = []
    if override:
        candidates.append(Path(override).expanduser().resolve())

    start = Path.cwd() if start is None else Path(start)
    candidates.extend([start, *start.parents])

    for candidate in candidates:
        if candidate.name == "g25":
            return candidate.resolve()
        sibling_g25 = candidate / "g25"
        if sibling_g25.exists() and sibling_g25.is_dir():
            return sibling_g25.resolve()

    created = start / "g25"
    created.mkdir(parents=True, exist_ok=True)
    return created.resolve()


def run(cmd, cwd: Path | None = None, env: dict | None = None, check: bool = True) -> subprocess.CompletedProcess:
    run_env = dict(os.environ)
    if env:
        run_env.update(env)
    if cwd is not None:
        run_env["PWD"] = str(cwd)
        run_env["OLDPWD"] = str(cwd)
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=run_env,
        check=False,
        capture_output=True,
        text=True,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {cmd}\nSTDOUT\n{result.stdout}\nSTDERR\n{result.stderr}"
        )
    return result


def update_env_from_env0(env_dump: bytes) -> None:
    for entry in env_dump.split(b"\0"):
        if not entry or b"=" not in entry:
            continue
        key, value = entry.split(b"=", 1)
        os.environ[key.decode(errors="ignore")] = value.decode(errors="ignore")


def load_fsl_module_if_needed() -> tuple[bool, str | None]:
    required_tools = ["mcflirt", "feat", "fslnvols"]
    if all(shutil.which(tool) for tool in required_tools):
        return True, "already-on-path"

    attempts = [
        ("ml fsl", "type ml >/dev/null 2>&1 && ml fsl"),
        ("module load fsl", "type module >/dev/null 2>&1 && module load fsl"),
    ]
    for label, shell_cmd in attempts:
        result = subprocess.run(
            ["bash", "-lc", f"{shell_cmd} >/dev/null 2>&1 && env -0"],
            capture_output=True,
            check=False,
        )
        if result.returncode != 0 or not result.stdout:
            continue
        update_env_from_env0(result.stdout)
        if all(shutil.which(tool) for tool in required_tools):
            return True, label

    return False, None


def gql(query: str, variables: dict) -> dict:
    payload = json.dumps({"query": query, "variables": variables}).encode("utf-8")
    req = urllib.request.Request(
        OPENNEURO_API,
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, context=CTX) as resp:
        return json.load(resp)


def snapshot_files(tree: str | None = None) -> list[dict]:
    query = """
    query($datasetId: ID!, $tag: String!, $tree: String) {
      snapshot(datasetId: $datasetId, tag: $tag) {
        files(tree: $tree) {
          key
          filename
          directory
          urls
          size
        }
      }
    }
    """
    return gql(query, {"datasetId": DATASET_ID, "tag": SNAPSHOT_TAG, "tree": tree})["data"]["snapshot"]["files"]


def subject_directory_key(subject: str, root_rows: list[dict]) -> str:
    for row in root_rows:
        if row.get("directory") and row["filename"] == subject:
            return row["key"]
    raise FileNotFoundError(f"Missing subject directory for {subject}")


def child_directory_key(parent_key: str, dirname: str) -> str:
    rows = snapshot_files(parent_key)
    for row in rows:
        if row.get("directory") and row["filename"] == dirname:
            return row["key"]
    raise FileNotFoundError(f"Missing child directory {dirname} under {parent_key}")


def file_url_map(tree_key: str) -> dict[str, str]:
    rows = snapshot_files(tree_key)
    return {row["filename"]: row["urls"][0] for row in rows if row.get("urls")}


def is_valid_nifti_gz(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with gzip.open(path, "rb") as src:
            header = src.read(352)
            for _chunk in iter(lambda: src.read(1024 * 1024), b""):
                pass
    except OSError:
        return False
    if len(header) < 352:
        return False
    sizeof_hdr_le = struct.unpack("<I", header[:4])[0]
    sizeof_hdr_be = struct.unpack(">I", header[:4])[0]
    return sizeof_hdr_le == 348 or sizeof_hdr_be == 348


def download_url(url: str, destination: Path, validate_nifti: bool = False) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        if validate_nifti and not is_valid_nifti_gz(destination):
            destination.unlink()
        else:
            return destination

    with urllib.request.urlopen(url, context=CTX) as resp:
        destination.write_bytes(resp.read())

    if destination.stat().st_size == 0:
        raise RuntimeError(f"Downloaded empty file for {destination.name}")
    if validate_nifti and not is_valid_nifti_gz(destination):
        raise RuntimeError(f"Downloaded invalid gzipped NIfTI file: {destination}")
    return destination


def ensure_uncompressed_nifti(source_path: Path, destination_path: Path) -> Path:
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if destination_path.exists() and destination_path.stat().st_size > 0:
        if destination_path.stat().st_mtime >= source_path.stat().st_mtime:
            return destination_path
        destination_path.unlink()

    tmp_path = destination_path.with_name(destination_path.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    if source_path.name.endswith(".nii.gz"):
        with gzip.open(source_path, "rb") as src, tmp_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    else:
        shutil.copy2(source_path, tmp_path)

    if not tmp_path.exists() or tmp_path.stat().st_size == 0:
        raise RuntimeError(f"Failed to materialize runtime NIfTI for {source_path}")
    os.replace(tmp_path, destination_path)
    return destination_path


def write_if_missing(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(text)


def patch_task_metadata(path: Path) -> None:
    if not path.exists():
        return
    payload = json.loads(path.read_text())
    if str(payload.get("TaskName", "")).startswith("TODO") or not payload.get("TaskName"):
        payload["TaskName"] = "Passive Ad Viewing"
    path.write_text(json.dumps(payload, indent=2) + "\n")


def plot_orthogonal_slices(volume: np.ndarray, title: str, out_path: Path, cmap: str = "gray") -> None:
    volume = np.asarray(volume)
    x, y, z = [dim // 2 for dim in volume.shape[:3]]
    vmax = float(np.percentile(volume, 99))
    vmin = float(np.percentile(volume, 1))

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(np.rot90(volume[x, :, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title("Sagittal")
    axes[1].imshow(np.rot90(volume[:, y, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title("Coronal")
    axes[2].imshow(np.rot90(volume[:, :, z]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[2].set_title("Axial")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def plot_adview_events(events: pd.DataFrame, count_path: Path, timeline_path: Path) -> None:
    order = ["Every_day_products", "Gambling", "vmPFC", "attention_check", "missed_trial"]
    counts = events["derived_trial_type"].value_counts().reindex(order, fill_value=0)
    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind="bar", color=["#5B8FF9", "#F4664A", "#5D7092", "#61DDAA", "#9A60B4"], ax=ax)
    ax.set_title("AdView trial counts by derived event label")
    ax.set_xlabel("Derived trial type")
    ax.set_ylabel("Number of events")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()
    fig.savefig(count_path, dpi=150)
    plt.close(fig)

    y_map = {label: idx for idx, label in enumerate(order)}
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.scatter(events["onset"], [y_map[label] for label in events["derived_trial_type"]], s=28, c="#4C78A8")
    ax.set_yticks(list(y_map.values()))
    ax.set_yticklabels(order)
    ax.set_xlabel("Onset (s)")
    ax.set_title("AdView event timeline for the representative run")
    ax.grid(alpha=0.25, axis="x")
    fig.tight_layout()
    fig.savefig(timeline_path, dpi=150)
    plt.close(fig)


def derive_adview_trial_type(row: pd.Series) -> str:
    response = str(row.get("attention_response", "")).strip()
    is_attention = str(row.get("is_attention_check", "")).strip().lower() in {"1", "true", "yes"}
    if response == "no_response":
        return "missed_trial"
    if is_attention:
        return "attention_check"
    return str(row["trial_type"])


def convert_adview_events_to_evs(events_path: Path, ev_dir: Path, run_id: int = 1) -> list[dict[str, object]]:
    ev_dir.mkdir(parents=True, exist_ok=True)
    frame = pd.read_csv(events_path, sep="\t").copy()
    frame["derived_trial_type"] = frame.apply(derive_adview_trial_type, axis=1)

    labels = ["Every_day_products", "Gambling", "vmPFC", "attention_check", "missed_trial"]
    manifest = []
    for label in labels:
        subset = frame.loc[frame["derived_trial_type"] == label, ["onset", "duration"]].copy()
        subset["amplitude"] = 1.0
        out_path = ev_dir / f"run-{run_id}_{label}.txt"
        if label == "missed_trial" and subset.empty:
            if out_path.exists():
                out_path.unlink()
            continue
        with out_path.open("w") as stream:
            for row in subset.itertuples(index=False):
                stream.write(f"{float(row.onset):.6f}\t{float(row.duration):.6f}\t1.0\n")
        manifest.append({"event": label, "rows": len(subset), "path": str(out_path)})
    return manifest


def find_first_command(candidates: list[str]) -> str | None:
    for candidate in candidates:
        command = shutil.which(candidate)
        if command:
            return command
    return None


def render_with_mricrogl(script_source: str, out_path: Path) -> tuple[Path, str, str | None]:
    mricrogl_cmd = find_first_command(["mricrogl", "MRIcroGL"])
    mricron_cmd = find_first_command(["mricron", "MRIcron"])
    if mricrogl_cmd is None:
        raise FileNotFoundError(
            "Could not find mricrogl/MRIcroGL on PATH. "
            "This notebook is written for Neurodesk environments where MRIcroGL is available as a command-line tool."
        )

    script_path = RUNTIME_ROOT / "mricrogl_render.py"
    script_path.write_text(script_source)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    attempts = [
        [mricrogl_cmd, str(script_path)],
        [mricrogl_cmd, "-s", str(script_path)],
    ]
    last_error = None
    for cmd in attempts:
        result = run(cmd, cwd=RUNTIME_ROOT, check=False)
        if out_path.exists() and out_path.stat().st_size > 0:
            return out_path, mricrogl_cmd, mricron_cmd
        last_error = f"Tried {cmd}\nSTDOUT\n{result.stdout}\nSTDERR\n{result.stderr}"
    raise RuntimeError(f"MRIcroGL did not create {out_path}.\n{last_error}")


def choose_runtime_paths(project_root: Path, output_dir: Path) -> tuple[Path, Path, str]:
    if " " not in str(project_root):
        runtime_root = output_dir / "_runtime"
        runtime_root.mkdir(parents=True, exist_ok=True)
        return runtime_root, project_root, "project-local"

    runtime_root = Path("/tmp/g25_adview_runtime")
    runtime_root.mkdir(parents=True, exist_ok=True)
    runtime_project = runtime_root / project_root.name
    if runtime_project.exists() or runtime_project.is_symlink():
        if runtime_project.resolve() != project_root.resolve():
            runtime_project.unlink()
    if not runtime_project.exists():
        runtime_project.symlink_to(project_root, target_is_directory=True)
    return runtime_root, runtime_project, "tmp-symlink"


PROJECT_ROOT = find_or_create_project_root(override=PROJECT_ROOT_OVERRIDE)
BIDS_ROOT = PROJECT_ROOT / "bids"
CODE_DIR = PROJECT_ROOT / "code"
TEMPLATE_DIR = PROJECT_ROOT / "templates"
DERIV_ROOT = PROJECT_ROOT / "derivatives" / "fsl"
EV_ROOT = DERIV_ROOT / "EVfiles"
OUTPUT_DIR = PROJECT_ROOT / "output" / "jupyter-notebook"
CACHE_DIR = OUTPUT_DIR / "_cache"
FIG_DIR = OUTPUT_DIR / "figures"
FEAT_DEMO_DIR = OUTPUT_DIR / "feat-demo"
CONF_DIR = FEAT_DEMO_DIR / "confounds"
FEAT_FIG_DIR = FEAT_DEMO_DIR / "figures"
WORKSPACE_SUMMARY_PATH = OUTPUT_DIR / "g25-adview-workspace-summary.json"
ADVIEW_TASK_JSON = BIDS_ROOT / "task-adview_bold.json"
ADVIEW_TEMPLATE_PATH = TEMPLATE_DIR / "L1_task-adview_model-1_type-act.fsf"

for path in [BIDS_ROOT, CODE_DIR, TEMPLATE_DIR, EV_ROOT, OUTPUT_DIR, CACHE_DIR, FIG_DIR, FEAT_DEMO_DIR, CONF_DIR, FEAT_FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

write_if_missing(PROJECT_ROOT / "README.md", WORKSPACE_README)
write_if_missing(CODE_DIR / "README.md", CODE_README)
write_if_missing(ADVIEW_TEMPLATE_PATH, TEMPLATE_ADVIEW_FSF)

RUNTIME_ROOT, RUNTIME_PROJECT, RUNTIME_MODE = choose_runtime_paths(PROJECT_ROOT, OUTPUT_DIR)
FSL_READY, FSL_LOAD_METHOD = load_fsl_module_if_needed()
MGL_CMD = find_first_command(["mricrogl", "MRIcroGL"])
MCRON_CMD = find_first_command(["mricron", "MRIcron"])

print(f"Project root: {PROJECT_ROOT}")
print(f"BIDS root: {BIDS_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Runtime root: {RUNTIME_ROOT}")
print(f"Runtime mode: {RUNTIME_MODE}")
print(f"FSL available: {FSL_READY}")
print(f"FSL load method: {FSL_LOAD_METHOD}")
print(f"mcflirt command: {find_first_command(['mcflirt'])}")
print(f"feat command: {find_first_command(['feat'])}")
print(f"MRIcroGL command: {MGL_CMD}")
print(f"MRIcron command: {MCRON_CMD}")


## Step 1 - Create a standalone `g25/` workspace and download AdView files from OpenNeuro

Purpose:
- This homework should run in Neurodesk without cloning a repository.
- This step creates the minimal `g25/` folder structure and downloads only the metadata, representative imaging files, and AdView event files needed for the rest of the notebook.

What the code is doing:
- Query the `ds007486` `v1.0.0` snapshot.
- Detect which approved subjects actually have `adview` in the snapshot.
- Download the root metadata files and `task-adview_bold.json`.
- Download `adview` `events.tsv` and `events.json` for every subject that has AdView.
- Download one representative T1w image and one representative AdView BOLD run for `sub-11039`.

QC checkpoint:
- The notebook should report a local `g25/` folder.
- The AdView availability table should show `sub-11028`, `sub-11039`, and `sub-11450`.
- The representative anatomy, BOLD, and event files should exist locally after this cell runs.


In [ ]:
root_rows = snapshot_files()
root_url_map = {row["filename"]: row["urls"][0] for row in root_rows if row.get("urls")}

for filename in ["README", "dataset_description.json", "participants.tsv", "participants.json", "task-adview_bold.json"]:
    if filename in root_url_map:
        download_url(root_url_map[filename], BIDS_ROOT / filename, validate_nifti=False)
patch_task_metadata(ADVIEW_TASK_JSON)

adview_rows = []
subject_func_maps = {}
available_adview_subjects = []
for subject in APPROVED_SUBJECTS:
    subject_key = subject_directory_key(subject, root_rows)
    anat_key = child_directory_key(subject_key, "anat")
    func_key = child_directory_key(subject_key, "func")
    anat_map = file_url_map(anat_key)
    func_map = file_url_map(func_key)
    subject_func_maps[subject] = {"anat": anat_map, "func": func_map}

    events_tsv = f"{subject}_task-adview_run-1_events.tsv"
    events_json = f"{subject}_task-adview_run-1_events.json"
    has_adview = events_tsv in func_map
    adview_rows.append({"subject": subject, "has_adview": has_adview, "events_file": events_tsv if has_adview else ""})

    if has_adview:
        available_adview_subjects.append(subject)
        subject_func_dir = BIDS_ROOT / subject / "func"
        download_url(func_map[events_tsv], subject_func_dir / events_tsv, validate_nifti=False)
        if events_json in func_map:
            download_url(func_map[events_json], subject_func_dir / events_json, validate_nifti=False)

adview_subject_table = pd.DataFrame(adview_rows)
available_subjects_csv = OUTPUT_DIR / "g25-adview-available-subjects.csv"
adview_subject_table.to_csv(available_subjects_csv, index=False)

if REPRESENTATIVE_SUBJECT not in available_adview_subjects:
    raise RuntimeError(f"{REPRESENTATIVE_SUBJECT} is not available for adview in this snapshot.")

anat_filename = f"{REPRESENTATIVE_SUBJECT}_T1w.nii.gz"
bold_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_echo-{ECHO}_part-mag_bold.nii.gz"
bold_json_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_echo-{ECHO}_part-mag_bold.json"
events_filename = f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_events.tsv"

anat_map = subject_func_maps[REPRESENTATIVE_SUBJECT]["anat"]
func_map = subject_func_maps[REPRESENTATIVE_SUBJECT]["func"]
rep_cache_dir = CACHE_DIR / "download-visualize" / REPRESENTATIVE_SUBJECT
rep_cache_dir.mkdir(parents=True, exist_ok=True)

anat_path = download_url(anat_map[anat_filename], rep_cache_dir / anat_filename, validate_nifti=True)
bold_path = download_url(func_map[bold_filename], rep_cache_dir / bold_filename, validate_nifti=True)
bold_json_path = download_url(func_map[bold_json_filename], rep_cache_dir / bold_json_filename, validate_nifti=False)
local_events_path = BIDS_ROOT / REPRESENTATIVE_SUBJECT / "func" / events_filename
bold_meta = json.loads(bold_json_path.read_text())

workspace_summary = {
    "project_root": str(PROJECT_ROOT),
    "available_adview_subjects": available_adview_subjects,
    "representative_subject": REPRESENTATIVE_SUBJECT,
    "task": TASK,
    "run": RUN,
}
WORKSPACE_SUMMARY_PATH.write_text(json.dumps(workspace_summary, indent=2) + "\n")

summary = pd.DataFrame([
    {"artifact": "Workspace README", "path": str(PROJECT_ROOT / "README.md"), "exists": (PROJECT_ROOT / "README.md").exists()},
    {"artifact": "Task sidecar", "path": str(ADVIEW_TASK_JSON), "exists": ADVIEW_TASK_JSON.exists()},
    {"artifact": "Representative T1w", "path": str(anat_path), "exists": anat_path.exists()},
    {"artifact": "Representative AdView BOLD", "path": str(bold_path), "exists": bold_path.exists()},
    {"artifact": "Representative AdView BOLD JSON", "path": str(bold_json_path), "exists": bold_json_path.exists()},
    {"artifact": "Representative AdView events", "path": str(local_events_path), "exists": local_events_path.exists()},
    {"artifact": "Available subjects CSV", "path": str(available_subjects_csv), "exists": available_subjects_csv.exists()},
    {"artifact": "Workspace summary JSON", "path": str(WORKSPACE_SUMMARY_PATH), "exists": WORKSPACE_SUMMARY_PATH.exists()},
])

display(adview_subject_table)
display(summary)
print(f"RepetitionTime from JSON: {bold_meta.get('RepetitionTime')}")

assert set(available_adview_subjects) == {"sub-11028", "sub-11039", "sub-11450"}
assert anat_path.exists()
assert bold_path.exists()
assert bold_json_path.exists()
assert local_events_path.exists()
assert float(bold_meta["RepetitionTime"]) > 0


## Step 2 - Inspect a representative anatomy file, AdView BOLD file, and AdView event structure

Purpose:
- Before building confounds or running FEAT, we confirm that the representative imaging files open correctly and that the AdView event table has the categories we expect.

What the code is doing:
- Load the representative T1w and AdView BOLD files with `nibabel`.
- Save one anatomy figure and one middle-volume BOLD figure.
- Load the AdView `events.tsv` file.
- Derive the event labels that will later become FSL EV files.
- Save an event-count plot and a timeline plot.

QC checkpoint:
- The T1w image should be 3D.
- The AdView BOLD file should be 4D.
- The AdView event table should include the three main stimulus classes and the derived attention/miss labels where present.
- The four saved PNGs should display inline below the cell.


In [ ]:
t1_img = nib.load(str(anat_path))
bold_img = nib.load(str(bold_path))
middle_bold_idx = bold_img.shape[3] // 2
middle_bold = np.asarray(bold_img.dataobj[..., middle_bold_idx])

events = pd.read_csv(local_events_path, sep="\t").copy()
events["derived_trial_type"] = events.apply(derive_adview_trial_type, axis=1)

t1_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_T1w.png"
bold_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_middle-volume.png"
event_count_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-counts.png"
event_timeline_png = FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-timeline.png"

print(f"T1w shape: {t1_img.shape}")
print(f"AdView BOLD shape: {bold_img.shape}")
print(f"Middle volume index: {middle_bold_idx}")
print("Derived AdView event counts:")
print(events["derived_trial_type"].value_counts().to_string())

assert len(t1_img.shape) == 3
assert len(bold_img.shape) == 4

plot_orthogonal_slices(np.asarray(t1_img.dataobj), f"{REPRESENTATIVE_SUBJECT} T1w", t1_png)
plot_orthogonal_slices(middle_bold, f"{REPRESENTATIVE_SUBJECT} AdView run-{RUN} middle volume", bold_png)
plot_adview_events(events, event_count_png, event_timeline_png)

display(events.head(12))
display(Image(filename=str(t1_png)))
display(Image(filename=str(bold_png)))
display(Image(filename=str(event_count_png)))
display(Image(filename=str(event_timeline_png)))


## Step 3 - Build FEAT-ready motion confounds for one representative AdView run

Purpose:
- This notebook does not require `fMRIPrep` for the homework.
- Instead, it uses `mcflirt` to estimate basic motion regressors for one representative AdView run so the FEAT model has standard motion nuisance terms.

What the code is doing:
- Materialize an uncompressed runtime NIfTI for FSL.
- Run `mcflirt` on the representative AdView BOLD run.
- Read the six rigid-body motion parameters from the `.par` file.
- Save a confounds TSV, framewise displacement text file, and FD plot.

QC checkpoint:
- The confounds table should have one row per BOLD volume and six columns.
- The FD plot should appear inline below the cell.
- The runtime and output paths should point inside the local `g25/` workspace.


In [ ]:
runtime_input_dir = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "_cache" / "download-visualize" / REPRESENTATIVE_SUBJECT
runtime_feat_dir = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "_runtime" / "feat-demo"
runtime_mc_dir = runtime_feat_dir / "mcflirt"
runtime_mc_dir.mkdir(parents=True, exist_ok=True)
CONF_DIR.mkdir(parents=True, exist_ok=True)

runtime_bold = ensure_uncompressed_nifti(
    bold_path,
    runtime_input_dir / bold_filename.replace(".nii.gz", ".nii"),
)
fd_metric_path = CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_fd-metric.txt"
fd_plot_path = CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_fd-plot.png"
mcflirt_base = runtime_mc_dir / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_mcf"
mcflirt_par = mcflirt_base.with_suffix(".par")
fsl_ready, fsl_load_method = load_fsl_module_if_needed()
mcflirt_cmd = find_first_command(["mcflirt"])
if fsl_load_method and fsl_load_method != "already-on-path":
    print(f"FSL load method for this cell: {fsl_load_method}")

if mcflirt_cmd is None:
    raise FileNotFoundError(
        "Could not find mcflirt on PATH, and automatic `ml fsl` / `module load fsl` did not succeed. "
        "Launch the notebook from a Neurodesk environment with FSL available."
    )

if not mcflirt_par.exists() or not mcflirt_base.with_suffix(".nii.gz").exists():
    for stale_path in [mcflirt_par, mcflirt_base.with_suffix(".nii.gz")]:
        if stale_path.exists():
            stale_path.unlink()
    run([
        mcflirt_cmd,
        "-in", str(runtime_bold),
        "-out", str(mcflirt_base),
        "-plots",
        "-mats",
        "-report",
    ])

confound_columns = ["rot_x", "rot_y", "rot_z", "trans_x", "trans_y", "trans_z"]
confounds = pd.read_csv(mcflirt_par, sep=r"\s+", header=None, names=confound_columns)
confounds_path = CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_mcflirt_confounds.tsv"
confounds.to_csv(confounds_path, sep="\t", index=False)

motion_values = confounds.to_numpy(dtype=float)
fd_values = np.zeros(motion_values.shape[0], dtype=float)
if motion_values.shape[0] > 1:
    diffs = np.abs(np.diff(motion_values, axis=0))
    fd_values[1:] = (50.0 * diffs[:, :3].sum(axis=1)) + diffs[:, 3:].sum(axis=1)
fd_metric = pd.DataFrame({"framewise_displacement": fd_values})
fd_metric.to_csv(fd_metric_path, header=False, index=False)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(fd_metric.index, fd_metric["framewise_displacement"], linewidth=1.2, color="steelblue")
ax.set_title("Motion outlier metric: fd")
ax.set_xlabel("time")
ax.set_ylabel("metric value")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(fd_plot_path, dpi=150)
plt.close(fig)

print(f"Runtime BOLD used by FSL: {runtime_bold}")
print(f"MCFLIRT corrected image: {mcflirt_base.with_suffix('.nii.gz')}")
print(f"Confound rows/columns: {confounds.shape}")
print(f"FD mean: {fd_metric.framewise_displacement.mean():.4f}")
print(f"FD max: {fd_metric.framewise_displacement.max():.4f}")
print(f"Confounds TSV: {confounds_path}")
print(f"FD metric file: {fd_metric_path}")
print(f"FD plot: {fd_plot_path}")

display(confounds.head())
display(Image(filename=str(fd_plot_path)))

assert confounds.shape[0] == bold_img.shape[3]
assert confounds.shape[1] == 6
assert fd_plot_path.exists()


## Step 4 - Convert all available AdView `events.tsv` files to FSL 3-column EV files

Purpose:
- This is the core cross-subject homework step for AdView.
- The goal is to show that the notebook can build FEAT-ready EV files directly from the BIDS event tables for every subject that has AdView in the snapshot.

What the code is doing:
- Loop over every subject with an AdView run in `v1.0.0`.
- Read the local `events.tsv` file for that subject.
- Derive AdView event labels from `trial_type`, `is_attention_check`, and `attention_response`.
- Write FSL 3-column EV files into `g25/derivatives/fsl/EVfiles/sub-<id>/adview/`.
- Save a manifest and a per-subject summary CSV.

QC checkpoint:
- The manifest should include `sub-11028`, `sub-11039`, and `sub-11450`.
- Each EV file should have exactly three columns.
- The representative subject should include `attention_check`, and `sub-11039` should also include `missed_trial`.


In [ ]:
ev_manifest_rows = []
summary_rows = []
for subject in available_adview_subjects:
    subject_events = BIDS_ROOT / subject / "func" / f"{subject}_task-{TASK}_run-1_events.tsv"
    subject_ev_dir = EV_ROOT / subject / TASK
    manifest_rows = convert_adview_events_to_evs(subject_events, subject_ev_dir, run_id=1)
    summary_rows.append({"subject": subject, "run": 1, "n_ev_files": len(manifest_rows)})
    for row in manifest_rows:
        ev_manifest_rows.append({
            "subject": subject,
            "run": 1,
            "event": row["event"],
            "rows": row["rows"],
            "path": row["path"],
        })

ev_summary = pd.DataFrame(summary_rows).sort_values(["subject", "run"]).reset_index(drop=True)
ev_manifest = pd.DataFrame(ev_manifest_rows).sort_values(["subject", "run", "event"]).reset_index(drop=True)
all_ev_summary_path = OUTPUT_DIR / "g25-adview-run-summary.csv"
all_ev_manifest_path = OUTPUT_DIR / "g25-adview-ev-manifest.csv"
ev_summary.to_csv(all_ev_summary_path, index=False)
ev_manifest.to_csv(all_ev_manifest_path, index=False)

print(f"AdView run summary: {all_ev_summary_path}")
print(f"AdView EV manifest: {all_ev_manifest_path}")
display(ev_summary)
display(ev_manifest)

preview_tables = []
for ev_path in sorted((EV_ROOT / REPRESENTATIVE_SUBJECT / TASK).glob("run-1_*.txt")):
    table = pd.read_csv(ev_path, sep=r"\s+", header=None, names=["onset", "duration", "amplitude"])
    table.insert(0, "file", ev_path.name)
    preview_tables.append(table.head())
if preview_tables:
    display(pd.concat(preview_tables, ignore_index=True))

assert set(ev_summary["subject"]) == set(available_adview_subjects)
assert all(pd.read_csv(Path(path), sep=r"\s+", header=None).shape[1] == 3 for path in ev_manifest["path"])
assert (EV_ROOT / "sub-11039" / TASK / "run-1_missed_trial.txt").exists()


## Step 5 - Run one representative first-level FEAT model for AdView

Purpose:
- This step demonstrates how the representative AdView run, its motion confounds, and the derived EV files fit together in a first-level FEAT model.

What the code is doing:
- Write the AdView FEAT template into the local `g25/templates/` folder if it is missing.
- Build a subject/run-specific `.fsf` file from that template.
- Point the template at the motion-corrected AdView BOLD file, the generated AdView EV files, and the motion confound `.par` file.
- Run `feat` and copy the important outputs into `g25/output/jupyter-notebook/feat-demo/`.

QC checkpoint:
- The FEAT design matrix figures should appear inline.
- `zstat1.nii.gz` should exist in the demo output folder.
- The `.fsf` file should be saved inside the notebook output area for inspection.


In [ ]:
runtime_feat_output = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "_runtime" / "feat-demo" / "feat" / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_model-1_demo"
runtime_feat_output.parent.mkdir(parents=True, exist_ok=True)
runtime_fsf_path = runtime_feat_output.parent / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_model-1_demo.fsf"
runtime_feat_output_dir = Path(str(runtime_feat_output) + ".feat")

feat_demo_files = {
    "fsf": FEAT_DEMO_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_model-1_demo.fsf",
    "design": FEAT_DEMO_DIR / "design.png",
    "design_cov": FEAT_DEMO_DIR / "design_cov.png",
    "example_func": FEAT_DEMO_DIR / "example_func.nii.gz",
    "zstat1": FEAT_DEMO_DIR / "zstat1.nii.gz",
    "report": FEAT_DEMO_DIR / "report.html",
    "report_log": FEAT_DEMO_DIR / "report_log.html",
    "feat1_log": FEAT_DEMO_DIR / "feat1.log",
}

if not feat_demo_files["zstat1"].exists():
    missed_trial_path = RUNTIME_PROJECT / "derivatives" / "fsl" / "EVfiles" / REPRESENTATIVE_SUBJECT / TASK / "run-1_missed_trial.txt"
    ev_shape = "3" if missed_trial_path.exists() else "10"
    template_text = ADVIEW_TEMPLATE_PATH.read_text()
    fsf_text = (
        template_text
        .replace("OUTPUT", str(runtime_feat_output))
        .replace("DATA", str(mcflirt_base.with_suffix(".nii.gz")))
        .replace("NVOLS", str(bold_img.shape[3]))
        .replace("EVDIR", str((RUNTIME_PROJECT / "derivatives" / "fsl" / "EVfiles" / REPRESENTATIVE_SUBJECT / TASK / "run-1")))
        .replace("MISSED_TRIAL", str(missed_trial_path))
        .replace("EV_SHAPE", ev_shape)
        .replace("SMOOTH", "5")
        .replace("CONFOUNDEVS", str(mcflirt_par))
    )
    runtime_fsf_path.write_text(fsf_text)

    feat_ready, feat_load_method = load_fsl_module_if_needed()
    feat_cmd = find_first_command(["feat"])
    if feat_load_method and feat_load_method != "already-on-path":
        print(f"FSL load method for FEAT cell: {feat_load_method}")
    if feat_cmd is None:
        raise FileNotFoundError(
            "Could not find feat on PATH, and automatic `ml fsl` / `module load fsl` did not succeed. "
            "Launch the notebook from a Neurodesk environment with FSL available."
        )
    run([feat_cmd, str(runtime_fsf_path)], cwd=runtime_feat_output.parent)

    shutil.copy2(runtime_fsf_path, feat_demo_files["fsf"])
    shutil.copy2(runtime_feat_output_dir / "design.png", feat_demo_files["design"])
    shutil.copy2(runtime_feat_output_dir / "design_cov.png", feat_demo_files["design_cov"])
    shutil.copy2(runtime_feat_output_dir / "example_func.nii.gz", feat_demo_files["example_func"])
    shutil.copy2(runtime_feat_output_dir / "stats" / "zstat1.nii.gz", feat_demo_files["zstat1"])
    shutil.copy2(runtime_feat_output_dir / "report.html", feat_demo_files["report"])
    shutil.copy2(runtime_feat_output_dir / "report_log.html", feat_demo_files["report_log"])
    shutil.copy2(runtime_feat_output_dir / "logs" / "feat1", feat_demo_files["feat1_log"])
    feat_mode = "ran-feat-now"
else:
    feat_mode = "reused-cached-feat-demo"

print(f"FEAT mode: {feat_mode}")
print(f"FSF file: {feat_demo_files['fsf']}")
print(f"Example func: {feat_demo_files['example_func']}")
print(f"zstat1: {feat_demo_files['zstat1']}")

display(Image(filename=str(feat_demo_files["design"])))
display(Image(filename=str(feat_demo_files["design_cov"])))

assert feat_demo_files["design"].exists()
assert feat_demo_files["example_func"].exists()
assert feat_demo_files["zstat1"].exists()


## Step 6 - Visualize the AdView FEAT result with Neurodesk MRIcroGL and embed the capture

Purpose:
- A saved FEAT result is easier to inspect when the activation map is rendered on top of the representative functional image.
- This final step demonstrates that the notebook can drive a Neurodesk visualization tool and save the output back into `g25/`.

What the code is doing:
- Load the AdView `zstat1` image.
- Find the peak voxel in the statistic map.
- Build a short MRIcroGL script that centers the view on that peak.
- Save the screenshot into the notebook output folder and display it inline.

QC checkpoint:
- The MRIcroGL script text should appear inline.
- The screenshot path should be reported.
- If `mricrogl` is available on PATH, the screenshot should display inline below the artifact table.


In [ ]:
zstat_img = nib.load(str(feat_demo_files["zstat1"]))
zstat_data = zstat_img.get_fdata()
peak_idx = np.unravel_index(np.nanargmax(np.abs(zstat_data)), zstat_data.shape)
peak_mm = nib.affines.apply_affine(zstat_img.affine, peak_idx)

mricrogl_png = FEAT_FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_zstat1_mricrogl.png"
runtime_example_func = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "feat-demo" / "example_func.nii.gz"
runtime_zstat1 = RUNTIME_PROJECT / "output" / "jupyter-notebook" / "feat-demo" / "zstat1.nii.gz"
mricrogl_rendered = False

mricrogl_script = textwrap.dedent(f'''import gl
gl.resetdefaults()
gl.loadimage({str(runtime_example_func)!r})
gl.overlayload({str(runtime_zstat1)!r})
gl.minmax(1, 2.5, 8)
gl.opacity(1, 80)
gl.colorname(1, '1red')
gl.orthoviewmm({peak_mm[0]:.3f}, {peak_mm[1]:.3f}, {peak_mm[2]:.3f})
gl.savebmp({str(mricrogl_png)!r})
gl.quit()
''')

print(f"Peak voxel index: {peak_idx}")
print(f"Peak millimeter coordinates: {tuple(round(v, 3) for v in peak_mm)}")
display(Markdown("**MRIcroGL Script**"))
display(Markdown(f"```python\n{mricrogl_script}```"))

if MGL_CMD is None:
    print("SKIP: MRIcroGL command not found on PATH in this environment. This step is expected to run in Neurodesk where mricrogl is available.")
    if MCRON_CMD:
        print(f"MRIcron is available at: {MCRON_CMD}")
elif (not mricrogl_png.exists()) or mricrogl_png.stat().st_size == 0:
    rendered_path, rendered_with, optional_mricron = render_with_mricrogl(mricrogl_script, mricrogl_png)
    mricrogl_rendered = True
    print(f"Rendered with: {rendered_with}")
    if optional_mricron:
        print(f"MRIcron also available at: {optional_mricron}")
    print(f"MRIcroGL screenshot: {rendered_path}")
else:
    print(f"Reusing existing MRIcroGL screenshot: {mricrogl_png}")

step6_summary = pd.DataFrame([
    {
        "artifact": "FEAT example_func",
        "path": str(runtime_example_func),
        "exists": runtime_example_func.exists(),
        "size_mb": round(runtime_example_func.stat().st_size / (1024 ** 2), 2) if runtime_example_func.exists() else np.nan,
    },
    {
        "artifact": "FEAT zstat1",
        "path": str(runtime_zstat1),
        "exists": runtime_zstat1.exists(),
        "size_mb": round(runtime_zstat1.stat().st_size / (1024 ** 2), 2) if runtime_zstat1.exists() else np.nan,
    },
    {
        "artifact": "MRIcroGL screenshot",
        "path": str(mricrogl_png),
        "exists": mricrogl_png.exists(),
        "size_mb": round(mricrogl_png.stat().st_size / (1024 ** 2), 2) if mricrogl_png.exists() else np.nan,
    },
])
display(Markdown("**Step 6 Artifacts**"))
display(step6_summary)

if mricrogl_png.exists() and mricrogl_png.stat().st_size > 0:
    display(Image(filename=str(mricrogl_png)))


## Troubleshooting and summary

If this notebook is run in a fresh Neurodesk directory, it should create `./g25` automatically.
If FSL is not already on `PATH`, launch Jupyter from a terminal where `ml fsl` succeeds.
The FEAT and MRIcroGL sections depend on FSL and MRIcroGL being available in the runtime environment.


In [ ]:
summary_rows = [
    {"artifact": "Workspace summary JSON", "path": str(WORKSPACE_SUMMARY_PATH)},
    {"artifact": "Available AdView subjects CSV", "path": str(OUTPUT_DIR / "g25-adview-available-subjects.csv")},
    {"artifact": "Representative T1w PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_T1w.png")},
    {"artifact": "Representative AdView BOLD PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_middle-volume.png")},
    {"artifact": "AdView event-count PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-counts.png")},
    {"artifact": "AdView event timeline PNG", "path": str(FIG_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_event-timeline.png")},
    {"artifact": "MCFLIRT confounds TSV", "path": str(CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_mcflirt_confounds.tsv")},
    {"artifact": "FD QC plot", "path": str(CONF_DIR / f"{REPRESENTATIVE_SUBJECT}_task-{TASK}_run-{RUN}_fd-plot.png")},
    {"artifact": "All-subject AdView EV summary", "path": str(OUTPUT_DIR / "g25-adview-run-summary.csv")},
    {"artifact": "All-subject AdView EV manifest", "path": str(OUTPUT_DIR / "g25-adview-ev-manifest.csv")},
    {"artifact": "All-subject AdView EV root", "path": str(EV_ROOT)},
    {"artifact": "FEAT FSF file", "path": str(feat_demo_files["fsf"])},
    {"artifact": "FEAT design matrix PNG", "path": str(feat_demo_files["design"])},
    {"artifact": "FEAT zstat1", "path": str(feat_demo_files["zstat1"])},
]
if "mricrogl_png" in globals() and mricrogl_png.exists():
    summary_rows.append({"artifact": "MRIcroGL screenshot", "path": str(mricrogl_png)})
summary = pd.DataFrame(summary_rows)
display(summary)
